# M6-T7 — Artifact Registry

**Owner:** Sanjeewa Narayana  
**Depends on:** M6-T2, M6-T3, M6-T4 — all artifacts must be uploaded to S3.

Generates `registry.json` — a versioned manifest of every M6 model artifact with
S3 paths, SHA-256 checksums, and test metrics. Uploads it to S3 so M7 can
discover and download artifacts without hard-coding paths.

| | |
|---|---|
| **Out** | `s3://email-security-pipeline-datasets/models/artifacts/registry.json` |
| **Schema doc** | `docs/m6-artifact-registry.md` |

In [1]:
import hashlib
import json
import shutil
import subprocess
from datetime import date
from pathlib import Path

S3_ARTIFACTS = 's3://email-security-pipeline-datasets/models/artifacts'
S3_PROFILE   = 'lab-user'  # change if needed, e.g. 'aws-lab', 'deploy-user'
REGISTRY_KEY = f'{S3_ARTIFACTS}/registry.json'

aws = shutil.which('aws') or '/usr/local/bin/aws'

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def s3_upload(local: Path, s3_path: str) -> None:
    r = subprocess.run(
        [aws, 's3', 'cp', str(local), s3_path, '--profile', S3_PROFILE],
        capture_output=True, text=True,
    )
    if r.returncode == 0:
        print(f'  ✅ {local.name} → {s3_path}')
    else:
        print(f'  ⚠️  upload failed: {r.stderr}')

print('Setup complete.')

Setup complete.


## Step 1 — Locate local artifacts and compute SHA-256

In [2]:
# Resolve models/ dir — works from project root or notebooks/M6/
_candidates = [Path('models'), Path('../../models'), Path('notebooks/M6/models')]
MODELS = next((p for p in _candidates if (p / 'email_linearsvc.joblib').exists()), None)
assert MODELS is not None, 'models/ dir not found — ensure artifacts are present locally'
print(f'Artifacts dir: {MODELS.resolve()}')

ARTIFACTS = [
    {
        'name':    'email_linearsvc',
        'track':   'email',
        'file':    'email_linearsvc.joblib',
        's3':      f'{S3_ARTIFACTS}/email/email_linearsvc.joblib',
        'metrics': {'f1': 0.9903, 'precision': 0.9900, 'recall': 0.9900, 'roc_auc': 0.9991},
        'notes':   'LinearSVC + TF-IDF (20k features, bigrams) + 5 numeric features',
    },
    {
        'name':    'url_charcnn',
        'track':   'url',
        'file':    'url_charcnn.keras',
        's3':      f'{S3_ARTIFACTS}/url/url_charcnn.keras',
        'metrics': {'f1': 0.9755, 'precision': 0.9769, 'recall': 0.9740, 'roc_auc': 0.9979},
        'notes':   'Char-CNN: Embedding(331,32)→Conv1D×3→GlobalMaxPool→Dropout(0.4)→Dense(1)',
    },
    {
        'name':    'url_char_vocab',
        'track':   'url',
        'file':    'url_char_vocab.json',
        's3':      f'{S3_ARTIFACTS}/url/url_char_vocab.json',
        'metrics': None,
        'notes':   'char2idx map + MAXLEN + VOCAB — required alongside url_charcnn.keras for inference',
    },
    {
        'name':    'url_rf',
        'track':   'url',
        'file':    'url_rf.joblib',
        's3':      f'{S3_ARTIFACTS}/url/url_rf.joblib',
        'metrics': {'f1': 0.8660, 'precision': 0.8755, 'recall': 0.8567, 'roc_auc': 0.9648},
        'notes':   'RandomForest(n_estimators=300) fallback — CPU-only, no TF dependency',
    },
]

for a in ARTIFACTS:
    path = MODELS / a['file']
    if path.exists():
        a['sha256'] = sha256_file(path)
        a['size_bytes'] = path.stat().st_size
        print(f"  {a['file']:<30}  {a['sha256'][:16]}...  ({a['size_bytes']/1e6:.1f} MB)")
    else:
        a['sha256'] = None
        a['size_bytes'] = None
        print(f"  {a['file']:<30}  ⚠️  not found locally — SHA-256 will be None")

Artifacts dir: /Users/nara/Documents/EDU/Sem2/CYT300/Project/models
  email_linearsvc.joblib          b7acb294c734cd54...  (0.9 MB)
  url_charcnn.keras               d1df114f8f4016ac...  (1.1 MB)
  url_char_vocab.json             b791f91244f15c87...  (0.0 MB)
  url_rf.joblib                   ⚠️  not found locally — SHA-256 will be None


## Step 2 — Build and save registry.json

In [3]:
registry = {
    'version':    'm6',
    'created':    str(date.today()),
    'splits':     's3://email-security-pipeline-datasets/datasets/processed/splits/',
    'artifacts':  [
        {k: v for k, v in a.items() if k != 'file'}
        for a in ARTIFACTS
    ],
}

registry_path = Path('registry.json')
registry_path.write_text(json.dumps(registry, indent=2))
print(json.dumps(registry, indent=2))

{
  "version": "m6",
  "created": "2026-06-22",
  "splits": "s3://email-security-pipeline-datasets/datasets/processed/splits/",
  "artifacts": [
    {
      "name": "email_linearsvc",
      "track": "email",
      "s3": "s3://email-security-pipeline-datasets/models/artifacts/email/email_linearsvc.joblib",
      "metrics": {
        "f1": 0.9903,
        "precision": 0.99,
        "recall": 0.99,
        "roc_auc": 0.9991
      },
      "notes": "LinearSVC + TF-IDF (20k features, bigrams) + 5 numeric features",
      "sha256": "b7acb294c734cd54d9cd445a4d7d139d65433002b8c7f02fae7cd10ddf857ffc",
      "size_bytes": 948253
    },
    {
      "name": "url_charcnn",
      "track": "url",
      "s3": "s3://email-security-pipeline-datasets/models/artifacts/url/url_charcnn.keras",
      "metrics": {
        "f1": 0.9755,
        "precision": 0.9769,
        "recall": 0.974,
        "roc_auc": 0.9979
      },
      "notes": "Char-CNN: Embedding(331,32)\u2192Conv1D\u00d73\u2192GlobalMaxPool\u2192

## Step 3 — Upload registry.json to S3

In [4]:
s3_upload(registry_path, REGISTRY_KEY)
print(f'\nVerify: aws s3 cp {REGISTRY_KEY} - --profile {S3_PROFILE}')

  ✅ registry.json → s3://email-security-pipeline-datasets/models/artifacts/registry.json

Verify: aws s3 cp s3://email-security-pipeline-datasets/models/artifacts/registry.json - --profile lab-user


## Summary

In [5]:
print('=' * 60)
print('M6-T7 ARTIFACT REGISTRY SUMMARY')
print('=' * 60)
print(f'Registry : {REGISTRY_KEY}')
print(f'Version  : {registry["version"]}  |  Created: {registry["created"]}')
print()
for a in registry['artifacts']:
    sha = a['sha256'][:16] + '...' if a['sha256'] else 'N/A'
    mb  = f"{a['size_bytes']/1e6:.1f} MB" if a['size_bytes'] else 'N/A'
    print(f"  {a['name']:<20}  track={a['track']:<6}  sha256={sha}  {mb}")
print()
print('Consumed by: M7 inference service (reads registry.json to resolve artifact paths)')
print('=' * 60)

M6-T7 ARTIFACT REGISTRY SUMMARY
Registry : s3://email-security-pipeline-datasets/models/artifacts/registry.json
Version  : m6  |  Created: 2026-06-22

  email_linearsvc       track=email   sha256=b7acb294c734cd54...  0.9 MB
  url_charcnn           track=url     sha256=d1df114f8f4016ac...  1.1 MB
  url_char_vocab        track=url     sha256=b791f91244f15c87...  0.0 MB
  url_rf                track=url     sha256=N/A  N/A

Consumed by: M7 inference service (reads registry.json to resolve artifact paths)
